# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [3]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %uv pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    %uv pip install --no-deps --upgrade "torchao>=0.16.0"
%uv pip install transformers==4.56.2
%uv pip install --no-deps trl==0.22.2

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 409.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 303.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 538.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 314.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 553.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 682.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 357.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 269.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 

In [4]:
%uv pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

Using Python 3.12.6 environment at: /usr/local
Resolved 30 packages in 268ms
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
einops     ------------------------------ 32.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 32.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 48.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 64.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 64.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 64.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 64.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 64.00 KiB/64.10 KiB
⠙ Preparing packages... (0/2)
einops     ------------------------------ 64.00 KiB/64.10 KiB
⠙ Pre

In [ ]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA nội bộ của Torch: {torch.version.cuda}")

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from unsloth import FastVisionModel, is_bfloat16_supported, UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from huggingface_hub.utils import logging, disable_progress_bars

# 2. Tắt hoàn toàn tất cả các thanh tiến trình (progress bar) của Hugging Face trên toàn hệ thống
disable_progress_bars()
logging.set_verbosity_error() 
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# ==========================================
# 1. CẤU HÌNH THAM SỐ
# ==========================================
HF_REPO_ID = "rimine/ct-rate-medgemma-ready"
MODEL_NAME = "unsloth/medgemma-1.5-4b-it" 
MAX_SLICES = 85

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH VỚI UNSLOTH (4-BIT)
# ==========================================
print("Loading model and processor...")
model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=False, # Bắt buộc True để tiết kiệm VRAM cho 85 ảnh
    use_gradient_checkpointing="unsloth", 
    attn_implementation = "flash_attention_2"
)
model.config.max_position_embeddings = 25808
processor.tokenizer.model_max_length = 25808
model.max_seq_length = 25808

# Cấu hình LoRA (Chỉ fine-tune phần ngôn ngữ và lớp kết nối Vision-Text
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision=False,       # Tạm thời đóng băng Vision Tower để tiết kiệm VRAM
    finetune_language=True,      # Fine-tune LLM Gemma
    finetune_attention_modules=True,
    finetune_mlp=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
)

# ==========================================
# 3. HÀM LAZY-LOADING DỮ LIỆU TỪ HUGGING FACE
# ==========================================
def lazy_load_and_format(examples):
    """
    Hàm này chỉ chạy khi Dataloader gọi đến.
    Nó tải file NPZ, giải nén thành list ảnh PIL và format thành chuẩn Chat Template.
    """
    batch_messages = []
    batch_images = []
    
    # zip qua từng thuộc tính của batch
    for npz_path, prompt, findings, impressions in zip(
        examples["npz_path"], examples["prompt"], examples["findings"], examples["impressions"]
    ):
        # 1. Tải file NPZ từ Hub về Cache cục bộ
        local_path = hf_hub_download(
            repo_id=HF_REPO_ID, 
            filename=npz_path, 
            repo_type="dataset"
        )
        
        # 2. Đọc file NPZ và trích xuất ma trận ảnh
        npz_data = np.load(local_path)
        # Giả định ma trận ảnh nằm ở key đầu tiên (vd: 'arr_0')
        img_array = npz_data[npz_data.files[0]] 
        
        # 3. Chuyển đổi ma trận uint8 thành list các đối tượng PIL Image
        # Cấu trúc img_array mong đợi: (Slices, Height, Width, Channels)
        slices = [Image.fromarray(img_array[i]) for i in range(img_array.shape[0])]
        
        # 4. Định dạng tin nhắn (Prompt)
        # Bơm các tag <image> tương ứng với số lượng lát cắt
        content_user = [{"type": "image"} for _ in range(len(slices))]
        content_user.append({"type": "text", "text": prompt})
        
        # Output mong đợi từ mô hình
        output_text = f"Findings:\n{findings}\n\nImpressions:\n{impressions}"
        
        messages = [
            {"role": "user", "content": content_user},
            {"role": "assistant", "content": [{"type": "text", "text": output_text}]}
        ]
        
        batch_messages.append(messages)
        batch_images.append(slices)
        
    return {"messages": batch_messages, "images": batch_images}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model and processor...
==((====))==  Unsloth 2026.5.8: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA B200. Num GPUs = 1. Max memory: 178.351 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 10.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
# ==========================================
# 4. LOAD DATASET VÀ GẮN TRANSFORM
# ==========================================
print("Loading dataset...")
dataset = load_dataset(HF_REPO_ID, split="train")

# set_transform ĐẢM BẢO việc tải ảnh chỉ diễn ra on-the-fly, không ngốn RAM
dataset.set_transform(lazy_load_and_format)

Loading dataset...


Resolving data files:   0%|          | 0/67 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/1675 [00:00<?, ? examples/s]

In [3]:
FastVisionModel.for_training(model) # Enable for training!
# ==========================================
# 5. CẤU HÌNH SFT TRAINER
# ==========================================
print("Initializing Trainer...")
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        packing = True,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        gradient_checkpointing = True,

        # use reentrant checkpointing
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        max_grad_norm = 0.3,              # max gradient norm based on QLoRA paper
        warmup_ratio = 0,
        max_steps = 30,
        #num_train_epochs = 2,          # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_torch_fused",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",             # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 25808
    )
)

Initializing Trainer...
Unsloth: Sample packing skipped (vision-language model detected).


[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [ ]:

# ==========================================
# 6. TIẾN HÀNH HUẤN LUYỆN VÀ LƯU TRỮ
# ==========================================
print("🚀 Bắt đầu huấn luyện...")
trainer.train()

# Lưu lại các trọng số LoRA (Adapters)
print("Saving model adapters...")
model.save_pretrained("medgemma_lora_adapters")
processor.save_pretrained("medgemma_lora_adapters")

# Nếu bạn muốn push thẳng lên Hugging Face:
# model.push_to_hub("rimine/medgemma-1.5-4b-ct-rate", token="HF_TOKEN")
# processor.push_to_hub("rimine/medgemma-1.5-4b-ct-rate", token="HF_TOKEN")

🚀 Bắt đầu huấn luyện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,675 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 19,248,896 of 4,319,328,368 (0.45% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.877300
2,2.452700
3,2.050500
4,2.024400
